**Contents**
1. Import Libraries and Read Dataset
1. Exploratory Data Analysis
1. Cohort Analysis
    * What is cohort analysis?
    * Cohort Analysis with Python
1. RFM Analysis for Customer Segmentation
    * What is RFM Analysis
    * RFM Analysis with Python
1. K-Means Clustering

## Import Libraries and Read Dataset

In [0]:
# pip install openpyxl
# %pip install Jinja2

# dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import datetime as dt

import os
# print(os.listdir("../input"))
# path=os.environ['USERPROFILE']+r'\OneDrive\BDA2'
# path=os.environ['USERPROFILE']+r'\Documents\BDA2'
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data"

# 文件路径应该位于 Volume 中 'BDA2_Data' 文件夹下
# VOLUME_DATA_FOLDER = "BDA2_Data/" 

# 1. 构造完整的 Volume 路径

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
path = (
    f"/Workspace/Users/{user}/bda_course/BDA2/"
)


In [0]:
df = pd.read_excel(path+r'data/Online Retail.xlsx')

## Exploratory Data Analysis

In [0]:
df.head()

In [0]:
df.info() 

In [0]:
df.describe()

In [0]:
df.corr()

In [0]:
def monthly(x):
    return dt.datetime(x.year, x.month, 1)

In [0]:
from datetime import datetime
monthly(datetime.fromisoformat('2010-12-26 08:26:00'))

1, divide and conquer , 2, help() , 3, ChatGPT , 4, google

In [0]:
df['InvoiceDate'].apply(monthly)

In [0]:
df['BillMonth'] = df['InvoiceDate'].apply(monthly)

In [0]:
df_sum = df.groupby('BillMonth').sum().drop('CustomerID', axis = 1)
df_sum = df_sum.rename(columns={'UnitPrice' : 'GrossProfit'})

plt.figure(figsize=(16,7))
sns.lineplot(x = df_sum.index, y = df_sum['GrossProfit'])
plt.show()

In [0]:
most_selling_products = df['Description'].value_counts()[:20]
plt.figure(figsize = (18,9))
sns.set_context("talk")
sns.barplot(x = most_selling_products.values, y = most_selling_products.index,palette = "tab20c_r")
plt.xticks(np.arange(0,2501,100),rotation = 90)
plt.xlabel("Amount")
plt.ylabel("Product")
plt.title("Most Selling 20 Products")

plt.show()

Data can explore more but I will directly jump into cohort analysis and customer segmentation

## Cohort Analysis

### What is cohort analysis?
<br>
Cohort analysis is a subset of behavioral analytics that takes the data from a given data set and rather than looking at all users as one unit, it breaks them into related groups for analysis. These related groups, or cohorts, usually share common characteristics or experiences within a defined time-span. Cohort analysis allows a company to “see patterns clearly across the life-cycle of a customer (or user), rather than slicing across all customers blindly without accounting for the natural cycle that a customer undergoes.” By seeing these patterns of time, a company can adapt and tailor its service to those specific cohorts. 

### Cohort Analysis with Python

In [0]:
g = df.groupby('CustomerID')['BillMonth']
df['CohortMonth'] = g.transform('min')
df.head()

In [0]:
df.query('BillMonth!=CohortMonth')

In [0]:
def get_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    return year, month

In [0]:
billYear, billMonth = get_int(df, 'BillMonth')
cohortYear, cohortMonth = get_int(df, 'CohortMonth')

In [0]:
diffYear = billYear - cohortYear
diffMonth = billMonth - cohortMonth

In [0]:
df['Month_Index'] = diffYear * 12 + diffMonth + 1

In [0]:
df.query('BillMonth!=CohortMonth')

In [0]:
df['CohortMonth'] = df['CohortMonth'].apply(dt.datetime.date)

In [0]:
g = df.groupby(['CohortMonth', 'Month_Index'])

In [0]:
cohortData = g['CustomerID'].apply(pd.Series.nunique).reset_index()
cohortCounts = cohortData.pivot(index = 'CohortMonth', columns = 'Month_Index', values = 'CustomerID')
cohortSizes = cohortCounts.iloc[:, 0]
retention = cohortCounts.divide(cohortSizes, axis = 0) * 100

In [0]:
g['CustomerID'].apply(pd.Series.nunique).reset_index()

In [0]:
cohortData.pivot(index = 'CohortMonth', columns = 'Month_Index', values = 'CustomerID')

In [0]:
cohortCounts.iloc[:, 0]

In [0]:
cohortCounts.divide(cohortSizes, axis = 0) * 100

In [0]:
retention.round(2)

In [0]:
month_list = ["Dec '10", "Jan '11", "Feb '11", "Mar '11", "Apr '11", "May '11", "Jun '11", "Jul '11", "Aug '11", "Sep '11", "Oct '11", "Nov '11", "Dec '11"]
plt.figure(figsize = (20,10))
plt.title('Retention by Monthly Cohorts')
sns.heatmap(retention.round(2), annot = True, cmap = "Blues", vmax = list(retention.max().sort_values(ascending = False))[1]-5, fmt = '.1f', linewidth = 0.3, yticklabels=month_list)
plt.show()

In [0]:
list(retention.max().sort_values(ascending = False))[1]

## RFM Analysis for Customer Segmentation

### What is RFM analysis?

RFM stands for Recency, Frequency, and Monetary value, each corresponding to some key customer trait. These RFM metrics are important indicators of a customer’s behavior because frequency and monetary value affects a customer’s lifetime value, and recency affects retention, a measure of engagement.

![](https://d35fo82fjcw0y8.cloudfront.net/2018/03/01013508/Incontent_image.png)

**Calculate RFM values** <br>
Let's calculate recency, frequency and monetary values. Also we will assume that we want cluster our customers into 5 segments.

In [0]:
calculating_date = max(df.InvoiceDate) + dt.timedelta(days = 1) # We assume that we are doing this analysis 1 day after from latest transaction on the data.

In [0]:
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

In [0]:
data = df.groupby(['CustomerID']).agg({
    'InvoiceDate': lambda x: (calculating_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalSum': 'sum'})

data.rename(columns={'InvoiceDate': 'Recency',
                         'InvoiceNo': 'Frequency',
                         'TotalSum': 'MonetaryValue'}, inplace=True)

data.head()

In [0]:
recency_labels = range(5, 0, -1)
frequency_labels = range(1, 6)

recency_groups = pd.qcut(data['Recency'], q=5, labels=recency_labels)
frequency_groups = pd.qcut(data['Frequency'], q=5, labels=frequency_labels)

data = data.assign(R=recency_groups.values, F=frequency_groups.values)

In [0]:
monetary_labels = range(1, 6)
monetary_groups = pd.qcut(data['MonetaryValue'], q=5, labels=monetary_labels)

data = data.assign(M=monetary_groups)

data['RFM_Score'] = data[['R','F','M']].sum(axis=1)
data['RFM_Score'].head()

In [0]:
def rfm_level(df):
    if df['RFM_Score'] >= 14:
        return 'Platinum Plus'
    elif ((df['RFM_Score'] >= 11) and (df['RFM_Score'] < 14)):
        return 'Platinum'
    elif ((df['RFM_Score'] >= 8) and (df['RFM_Score'] < 11)):
        return 'Gold'
    elif ((df['RFM_Score'] >= 6) and (df['RFM_Score'] < 8)):
        return 'Silver'
    else:
        return 'Bronze'

data['RFM_Level'] = data.apply(rfm_level, axis=1)

data.head()

In [0]:
rfm_level_agg = data.groupby('RFM_Level').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': ['mean', 'count']
}).round(1)

rfm_level_agg

## K-Means Clustering

Now we will try to cluster our customers with one of the most used clustering ML algorithm: K-Means! But before do that we have to know some details about K-Means. K-Means assumes your variables have symmetric distributions, they have same average values and same variance. So, we will prepare our data according to this assumptions. 

![image.png](attachment:image.png)

![image.png](attachment:image.png)

In [0]:
data_rfm = data[['Recency', 'Frequency', 'MonetaryValue']]

plt.figure(figsize = (17,20))
plt.subplot(3, 1, 1); sns.distplot(data_rfm['Recency'])
plt.subplot(3, 1, 2); sns.distplot(data_rfm['Frequency'])
plt.subplot(3, 1, 3); sns.distplot(data_rfm['MonetaryValue'])
plt.show()

In [0]:
data_rfm.min()

Since we will apply log transformation we have to make all values positive.

In [0]:
data_rfm_positive = data_rfm
data_rfm_positive.MonetaryValue = data_rfm.MonetaryValue + abs(data_rfm.MonetaryValue.min()) + 1

In [0]:
data_rfm_positive.min()

In [0]:
from sklearn.preprocessing import StandardScaler

data_log = np.log(data_rfm)
scaler = StandardScaler()
scaler.fit(data_log)
data_normalized = scaler.transform(data_log)
data_normalized = pd.DataFrame(data=data_normalized, index=data_rfm.index, columns=data_rfm.columns)

In [0]:
plt.figure(figsize = (17,20))
plt.subplot(3, 1, 1); sns.distplot(data_log['Recency'])
plt.subplot(3, 1, 2); sns.distplot(data_log['Frequency'])
plt.subplot(3, 1, 3); sns.distplot(data_log['MonetaryValue'])
plt.show()

In [0]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=1) 
kmeans.fit(data_normalized)
cluster_labels = kmeans.labels_

In [0]:
data_rfm_k5 = data_rfm.assign(Cluster=cluster_labels)
grouped = data_rfm_k5.groupby(['Cluster'])
grouped.agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': ['mean', 'count']
  }).round(1)

To find which 'k' value is more suitable for our data we will use elbow method.

In [0]:
sse = {}
for k in range(2, 15):  
    kmeans = KMeans(n_clusters=k, random_state=1)
    kmeans.fit(data_normalized)
    sse[k] = kmeans.inertia_ 

In [0]:
plt.figure(figsize=(18,9))

plt.title('The Elbow Method')
plt.xlabel('k')
plt.ylabel('SSE')
sns.pointplot(x=list(sse.keys()), y=list(sse.values()))
plt.show()

We can say 3 is the best k value.

In [0]:
kmeans = KMeans(n_clusters=3, random_state=1) 
kmeans.fit(data_normalized)
cluster_labels = kmeans.labels_
data_rfm_k3 = data_rfm.assign(Cluster=cluster_labels)
grouped = data_rfm_k3.groupby(['Cluster'])
grouped.agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': ['mean', 'count']
  }).round(1)

In [0]:
data_melt = pd.melt(
                    data_rfm_k3.reset_index(),               
                    id_vars=['CustomerID', 'Cluster'],
                    value_vars=['Recency', 'Frequency', 'MonetaryValue'], 
                    var_name='Metric', value_name='Value'
                    )

In [0]:
%pip install jinja2

In [0]:
data_melt.head(10)

In [0]:
# Snake Plot
plt.figure(figsize = (18,9))
plt.title('Snake plot of normalized variables')
plt.xlabel('Metric')
plt.ylabel('Value')
sns.lineplot(data=data_melt, x='Metric', y='Value', hue='Cluster')
plt.show()

**Calculate relative importance of each attribute**<br>
Now we will calculate the relative importance of the RFM values within each cluster.

In [0]:
cluster_avg = data_rfm_k3.groupby(['Cluster']).mean() 
population_avg = data_rfm.mean()
relative_imp = cluster_avg / population_avg - 1
relative_imp.round(2)

In [0]:
plt.figure(figsize=(13, 5))
plt.title('Relative importance of attributes')
sns.heatmap(data=relative_imp, annot=True, fmt='.2f', cmap='RdYlGn')
plt.show()

In [0]:


relative_imp.style.background_gradient(cmap='RdYlGn',axis=1)

### Tenure

In [0]:
tenure_list = []
for i in list(data_rfm.index):
    tenure_list.append((df.InvoiceDate.max() - df[(df.CustomerID == i)]['InvoiceDate'].min()).days + 1)

In [0]:
data_rfmt = data_rfm.assign(Tenure = tenure_list)
data_rfmt.min()

In [0]:
data_rfmt_log = np.log(data_rfmt)
scaler = StandardScaler()
scaler.fit(data_rfmt_log)
data_rfmt_normalized = scaler.transform(data_rfmt_log)

In [0]:
sse = {}
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=1).fit(data_rfmt_normalized)
    sse[k] = kmeans.inertia_ 

plt.figure(figsize = (13,7))
plt.title('The Elbow Method'); plt.xlabel('k'); plt.ylabel('SSE')
sns.pointplot(x=list(sse.keys()), y=list(sse.values()))
plt.show()

3 or 4 ? 

In [0]:
kmeans = KMeans(n_clusters=4, random_state=1) 
kmeans.fit(data_rfmt_normalized)
cluster_labels = kmeans.labels_

In [0]:
data_rfmt_k4 = data_rfmt.assign(Cluster=cluster_labels)
grouped = data_rfmt_k4.groupby(['Cluster'])
grouped.agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': 'mean',
    'Tenure': ['mean', 'count']
  }).round(1)